# Aula 06 - Do Script ao Produto: React, FastAPI e Interpretação com LLM

Módulo 08 AdmTech, Computação · Semana 10 · Sprint 5 · Prof. José Romualdo

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/josercf/2026-2B-M08/blob/main/notebooks/aula06.ipynb)

## O que este notebook é

Laboratório executável da Aula 06, que alimenta o artefato A5 (Simulador Interativo integrado à interface e Interpretação por LLM). A API é definida aqui dentro e testada com `TestClient`, sem subir servidor, o que funciona igual no Colab e no CI. A versão completa, com backend e frontend separados, está em `exemplos/aula06-api-carteira/` do repositório. Sem rede, a carga de dados usa o snapshot versionado em `dados/`, e a chamada ao LLM é pulada.

## 1. Dados do case

A mesma carga das aulas anteriores: o CSV vem do repositório clonado ao lado e, no Colab, da URL bruta do GitHub. `M08_OFFLINE=1` desliga a rede, que é como o CI roda.

In [ ]:
import os
from functools import lru_cache

import numpy as np
import pandas as pd

DADOS_LOCAL = os.path.join("..", "dados")
DADOS_BRUTOS = "https://raw.githubusercontent.com/josercf/2026-2B-M08/main/dados/"
OFFLINE = os.environ.get("M08_OFFLINE") == "1"


def ler_csv(nome, **kwargs):
    """Lê um CSV de dados/: do disco quando o repositório está clonado, da URL bruta no Colab."""
    local = os.path.join(DADOS_LOCAL, nome)
    if os.path.exists(local):
        return pd.read_csv(local, **kwargs)
    if OFFLINE:
        raise FileNotFoundError("dados/ não encontrado e M08_OFFLINE=1 desliga a rede")
    return pd.read_csv(DADOS_BRUTOS + nome, **kwargs)


PRECOS = ler_csv("precos_ajustados.csv", index_col="data", parse_dates=["data"])
CDI = ler_csv("cdi_diario.csv", index_col="data", parse_dates=["data"])["valor"]
print(len(PRECOS), "pregões e", PRECOS.shape[1], "ativos; offline =", OFFLINE)

## 2. Os módulos das sprints anteriores

No repositório do grupo, cada função abaixo mora no módulo do artefato que a criou (coletor do A1, métricas do A2, otimizador do A4), e a API faz `from carteira import pipeline` em vez de copiar código. Aqui elas ficam numa célula só porque o Colab abre um arquivo de cada vez. A função `calcular` não sabe nada de HTTP: recebe tickers e datas e devolve um dicionário, e por isso serve à API, ao notebook e ao teste.

O tratamento de ausência é o do módulo (ADR-009): a lacuna isolada recebe o último preço (`ffill`), nada é preenchido antes da primeira cotação, e os retornos são recortados na janela comum dos tickers pedidos. Um pedido sem GOLD11 não perde 2019 e 2020; com ela, a janela começa em 14/12/2020. A resposta declara o início, o fim e o número de retornos da janela.

In [ ]:
from scipy.optimize import minimize

PREGOES_ANO = 252
PROBLEMAS = {"resolvidos": 0}  # cada chamada ao SLSQP; é o que o cache economiza


class TickerDesconhecido(LookupError):
    pass


class PeriodoSemDados(ValueError):
    pass


def resolver(objetivo, n, restricoes=()):
    PROBLEMAS["resolvidos"] += 1
    r = minimize(objetivo, np.full(n, 1 / n), method="SLSQP", bounds=[(0, 1)] * n,
                 constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}, *restricoes],
                 options={"ftol": 1e-12, "maxiter": 500})
    pesos = np.clip(r.x, 0, None)
    return pesos / pesos.sum()


def calcular(tickers, inicio, fim, pontos=50):
    fora = [t for t in tickers if t not in PRECOS.columns]
    if fora:
        raise TickerDesconhecido(", ".join(fora))
    # ADR-009: ffill preenche a lacuna isolada e nunca o começo da série; o retorno
    # é calculado na série inteira e recortado na janela comum dos tickers pedidos.
    precos = PRECOS[list(tickers)].ffill()
    ret = precos.pct_change(fill_method=None).iloc[1:].loc[inicio:fim].dropna()
    if len(ret) < 60:
        raise PeriodoSemDados("a janela comum tem %d retornos; o mínimo é 60" % len(ret))
    mu, cov = ret.mean().values * PREGOES_ANO, ret.cov().values * PREGOES_ANO
    cdi = CDI.loc[ret.index[0]:ret.index[-1]] / 100  # CDI dos mesmos pregões da janela
    rf = float(np.prod(1 + cdi) ** (PREGOES_ANO / len(cdi)) - 1)
    minima = resolver(lambda w: w @ cov @ w, len(mu))
    sharpe = resolver(lambda w: -(w @ mu - rf) / np.sqrt(w @ cov @ w), len(mu))
    fronteira = []
    for alvo in np.linspace(minima @ mu, mu.max(), pontos):
        w = resolver(lambda w: w @ cov @ w, len(mu), ({"type": "eq", "fun": lambda w, a=alvo: w @ mu - a},))
        fronteira.append((float(np.sqrt(w @ cov @ w)), float(alvo)))
    corr = ret.corr()

    def carteira(w):
        retorno, vol = float(w @ mu), float(np.sqrt(w @ cov @ w))
        return {"pesos": dict(zip(tickers, map(float, w))), "retorno": retorno,
                "volatilidade": vol, "sharpe": (retorno - rf) / vol}

    return {"tickers": list(tickers), "inicio_janela": ret.index[0].date().isoformat(),
            "fim_janela": ret.index[-1].date().isoformat(), "pregoes": len(ret), "taxa_livre_risco": rf,
            "ativos": [{"ticker": t, "retorno": float(mu[i]),
                        "volatilidade": float(np.sqrt(cov[i, i])),
                        "correlacao_media": float((corr[t].sum() - 1) / (len(tickers) - 1))}
                       for i, t in enumerate(tickers)],
            "variancia_minima": carteira(minima), "maximo_sharpe": carteira(sharpe),
            "fronteira": fronteira}


TICKERS = ["PETR4", "VALE3", "ITUB4", "WEGE3", "IVVB11"]
r = calcular(TICKERS, "2019-01-01", "2026-08-31")
print("retornos diários:", r["pregoes"], "de", r["inicio_janela"], "a", r["fim_janela"],
      "| problemas resolvidos:", PROBLEMAS["resolvidos"])
pd.DataFrame({k: r[k]["pesos"] for k in ("variancia_minima", "maximo_sharpe")}).mul(100).round(2)

## 3. A API: uma rota por operação de negócio

A rota tem nome do que o gestor faz (`/carteiras/otimizar`), e o modelo Pydantic é o contrato: ele recusa a entrada errada antes de qualquer cálculo, com status 422 e a lista do que está errado. O que só o cálculo descobre, como um ticker fora da base, vira `HTTPException` com o status que o contrato declarou. O `lru_cache` guarda o resultado de cada consulta já feita.

No Colab, `fastapi` e `httpx` já vêm instalados. Fora dele, `pip install "fastapi[standard]"`.

In [ ]:
from datetime import date

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field, field_validator, model_validator

app = FastAPI(title="Simulador de carteira eficiente (notebook da Aula 06)")


class PedidoCarteira(BaseModel):
    tickers: list[str] = Field(min_length=2, max_length=15)
    inicio: date = date(2019, 1, 1)
    fim: date = date(2026, 8, 31)

    @field_validator("tickers")
    @classmethod
    def normalizar(cls, tickers):
        return [t.strip().upper() for t in tickers]

    @model_validator(mode="after")
    def conferir(self):
        if self.inicio >= self.fim:
            raise ValueError("inicio precisa ser anterior a fim")
        if len(set(self.tickers)) != len(self.tickers):
            raise ValueError("ticker repetido na lista")
        return self


@lru_cache(maxsize=64)
def calcular_com_cache(tickers, inicio, fim):
    try:
        return calcular(list(tickers), inicio, fim)
    except TickerDesconhecido as erro:
        raise HTTPException(404, "Ticker fora da base: %s" % erro)
    except PeriodoSemDados as erro:
        raise HTTPException(422, "Período sem dados suficientes: %s" % erro)


@app.post("/carteiras/otimizar")
def otimizar(pedido: PedidoCarteira):
    return calcular_com_cache(tuple(pedido.tickers), pedido.inicio.isoformat(), pedido.fim.isoformat())

## 4. Prática em dupla: a API testada sem servidor

O `TestClient` chama a API como um navegador chamaria, sem abrir porta nenhuma. Condutor no teclado, navegador conferindo; troquem aos 6 minutos.

a. Rodem a célula abaixo: um caso de sucesso (200) e um de entrada inválida (422).
b. Leiam o corpo do 422: que campo o Pydantic aponta, e com que mensagem?
c. Escrevam o caso do ticker que não existe. Antes de rodar, decidam em dupla qual status ele deve devolver.

In [ ]:
import warnings

# Aviso de depreciação interno do Starlette, sem efeito no teste.
warnings.filterwarnings("ignore", message="Using .httpx. with .starlette.testclient.")
from fastapi.testclient import TestClient

cliente = TestClient(app)
PEDIDO = {"tickers": TICKERS, "inicio": "2019-01-01", "fim": "2026-08-31"}

ok = cliente.post("/carteiras/otimizar", json=PEDIDO)
assert ok.status_code == 200
assert abs(sum(ok.json()["maximo_sharpe"]["pesos"].values()) - 1) < 1e-9
print("200:", ok.json()["pregoes"], "retornos diários e", len(ok.json()["fronteira"]), "pontos de fronteira")

invalido = cliente.post("/carteiras/otimizar", json=dict(PEDIDO, tickers=["PETR4"]))
assert invalido.status_code == 422
print("422:", invalido.json()["detail"][0]["loc"], invalido.json()["detail"][0]["msg"])

In [ ]:
# c. Caso do ticker que não existe: troquem None pelo status que a dupla decidiu.
status_esperado = None

resposta = cliente.post("/carteiras/otimizar", json=dict(PEDIDO, tickers=["PETR4", "XPTO3"]))
print(resposta.status_code, resposta.json())
if status_esperado is not None:
    assert resposta.status_code == status_esperado, "a API devolveu %d" % resposta.status_code

<details>
<summary>Resposta da dupla</summary>

O exemplo adota 404, com o ticker na mensagem: o pedido está bem formado (dois tickers, datas em ordem) e passa no Pydantic, e o que falta é o recurso pedido. O 422 também é defensável, porque o ticker vem no corpo do pedido. O que não pode faltar é a escolha escrita no contrato do `SPEC.md`, com o corpo de erro de cada caso.

</details>

## 5. Cache: a mesma consulta não resolve de novo

Cada pedido novo resolve 2 carteiras e 50 pontos de fronteira. Com o `lru_cache`, a segunda chamada igual devolve o resultado guardado sem nenhum problema de otimização. A chave do cache é a consulta inteira: trocar a data de início é outra consulta.

In [ ]:
calcular_com_cache.cache_clear()
for rotulo, pedido in [("A", PEDIDO), ("A de novo", PEDIDO), ("B, desde 2021", dict(PEDIDO, inicio="2021-01-01"))]:
    antes = PROBLEMAS["resolvidos"]
    cliente.post("/carteiras/otimizar", json=pedido).raise_for_status()
    print("%-14s problemas resolvidos: %d" % (rotulo, PROBLEMAS["resolvidos"] - antes))

## 6. O LLM como componente do produto

O prompt leva os números reais da carteira, em porcentagem com duas casas, e as regras de escrita: usar só esses números, declarar as limitações, não prometer retorno. A chamada é do servidor, com a chave em variável de ambiente. No Colab, guarde a chave em Secrets (ícone de chave na barra lateral) com o nome `OPENROUTER_API_KEY`; a célula abaixo tenta lê-la de lá. Sem chave ou com `M08_OFFLINE=1`, a chamada é pulada e o notebook segue.

In [ ]:
import json


def contexto(r):
    pct = lambda x: round(100 * x, 2)
    carteira = lambda c: {"pesos_pct": {t: pct(p) for t, p in c["pesos"].items() if p >= 0.0005},
                          "retorno_anual_pct": pct(c["retorno"]),
                          "volatilidade_anual_pct": pct(c["volatilidade"]),
                          "sharpe": round(c["sharpe"], 2)}
    return {"inicio_janela": r["inicio_janela"], "fim_janela": r["fim_janela"],
            "pregoes": r["pregoes"], "taxa_livre_risco_pct": pct(r["taxa_livre_risco"]),
            "ativos": [{"ticker": a["ticker"], "retorno_anual_pct": pct(a["retorno"]),
                        "volatilidade_anual_pct": pct(a["volatilidade"]),
                        "correlacao_media": round(a["correlacao_media"], 2)} for a in r["ativos"]],
            "variancia_minima": carteira(r["variancia_minima"]),
            "maximo_sharpe": carteira(r["maximo_sharpe"])}


INSTRUCOES = ("Você é analista de carteiras e escreve para um gestor financeiro que não programa. "
              "Use apenas os números do JSON abaixo, em porcentagem com duas casas. "
              "Termine com as limitações: os números vêm de uma janela passada e os pesos mudam com ela. "
              "Não recomende compra nem prometa retorno futuro.")
prompt = INSTRUCOES + "\n\nResultado da otimização:\n" + json.dumps(contexto(r), ensure_ascii=False, indent=1)
print(prompt[:900])

In [ ]:
def chave_do_llm():
    if OFFLINE:
        return None
    try:
        from google.colab import userdata  # só existe no Colab
        return userdata.get("OPENROUTER_API_KEY")
    except Exception:
        return os.environ.get("OPENROUTER_API_KEY")


chave = chave_do_llm()
texto_llm = None
if chave:
    import httpx

    resposta = httpx.post("https://openrouter.ai/api/v1/chat/completions", timeout=60,
                          headers={"Authorization": "Bearer " + chave},
                          json={"model": "openai/gpt-4o-mini", "temperature": 0.2,
                                "messages": [{"role": "user", "content": prompt}]})
    resposta.raise_for_status()
    texto_llm = resposta.json()["choices"][0]["message"]["content"]
    print(texto_llm)
else:
    print("sem chave ou offline: chamada ao LLM pulada")

## 7. Conferir cada número do texto

O texto abaixo é **um exercício**: foi escrito por este notebook a partir dos números reais, no estilo de um LLM, com três erros plantados. Nenhum modelo o gerou. Leiam em dupla e marquem os números suspeitos antes de rodar o verificador.

O verificador é uma conferência de presença: cada número do texto precisa existir em algum campo do JSON que foi no prompt. Ele pega número inventado e número alterado. Ele não pega número certo atribuído ao ativo errado, e por isso devolve também de onde veio cada número.

In [ ]:
import re

br = lambda x: ("%.2f" % x).replace(".", ",")
c = contexto(r)
s, m = c["maximo_sharpe"], c["variancia_minima"]
pesos = sorted(s["pesos_pct"].items(), key=lambda par: -par[1])
menor = min(c["ativos"], key=lambda a: a["correlacao_media"])
maior_m = max(m["pesos_pct"].items(), key=lambda par: par[1])
texto_exercicio = (
    "Com %s retornos diários e o CDI de %s%% ao ano como taxa livre de risco, a carteira de máximo "
    "Sharpe concentra %s. Ela tem retorno anual de %s%% com volatilidade de %s%%, e Sharpe de %s. "
    "A carteira de variância mínima reduz a volatilidade para %s%%, com %s%% em %s. A VALE3 é o ativo "
    "que mais diversifica, com correlação média de %s. Se o cenário se repetir, a carteira de máximo "
    "Sharpe deve render %s%% no próximo ano. Esses números vêm de uma janela passada, e os pesos mudam "
    "quando a janela muda."
    % ("{:,}".format(c["pregoes"]).replace(",", "."), br(c["taxa_livre_risco_pct"]),
       ", ".join("%s%% em %s" % (br(p), t) for t, p in pesos[:-1]) + " e %s%% em %s" % (br(pesos[-1][1]), pesos[-1][0]),
       br(s["retorno_anual_pct"]), br(s["volatilidade_anual_pct"] - 2), br(s["sharpe"]),
       br(m["volatilidade_anual_pct"]), br(maior_m[1]), maior_m[0], br(menor["correlacao_media"]), br(30)))
print(texto_exercicio)

In [ ]:
NUMERO = re.compile(r"(?<![\w,./])(\d{1,3}(?:\.\d{3})+|\d+)(?:,(\d+))?(?![\w/])(\s?%)?")


def folhas(valor, caminho=""):
    if isinstance(valor, dict):
        for k, v in valor.items():
            yield from folhas(v, caminho + "." + k if caminho else k)
    elif isinstance(valor, list):
        for item in valor:
            yield from folhas(item, "%s[%s]" % (caminho, item.get("ticker", "")))
    elif isinstance(valor, (int, float)):
        yield caminho, valor


def verificar_numeros(texto, ctx):
    todas = list(folhas(ctx))
    saida = []
    for casamento in NUMERO.finditer(texto):
        inteiro, decimal, pct = casamento.groups()
        valor = float(inteiro.replace(".", "") + ("." + decimal if decimal else ""))
        origem = [k for k, v in todas if ("pct" in k) == bool(pct) and abs(v - valor) < 0.005]
        saida.append({"trecho": casamento.group(0).strip(), "confere": bool(origem),
                      "origem": origem[0] if origem else None})
    return pd.DataFrame(saida)


tabela = verificar_numeros(texto_exercicio, c)
print(tabela["confere"].sum(), "de", len(tabela), "números conferem")
tabela

In [ ]:
# Quando houver chave, o mesmo verificador confere o texto de verdade do modelo.
if texto_llm:
    verificar_numeros(texto_llm, c)

### Pergunta para a dupla

O verificador aprovou o 0,16 atribuído à VALE3. Que regra mais estrita pegaria esse erro, e que custo ela teria? Pensem em exigir que o ticker citado na mesma frase seja o mesmo da `origem`.

<details>
<summary>Resposta</summary>

A coluna `origem` diz que 0,16 é a correlação média do IVVB11. Uma regra que procura o ticker na mesma frase e compara com a origem pega o erro, ao custo de depender de como o modelo escreve a frase: uma frase que cita dois ativos já confunde a regra. Por isso a verificação de presença roda sempre, e a leitura humana continua no fluxo.

</details>

## 8. Desafio para o A5

a. Acrescentem a rota `POST /carteiras/metricas`, que recebe `pesos` (dicionário ticker para peso) e devolve retorno, volatilidade e Sharpe dessa carteira.
b. Pesos que não somam 1 (com tolerância de 0,001) devolvem 422, com a soma recebida na mensagem.
c. Completem a célula de teste abaixo com um caso 200 e um caso 422, e levem os dois para o `SPEC.md` do grupo como casos de teste do contrato.

In [ ]:
# Teste do desafio: roda quando a rota existir.
rotas = {getattr(rota, "path", None) for rota in app.routes}
if "/carteiras/metricas" in rotas:
    pedido_pesos = {"tickers": TICKERS, "pesos": {"PETR4": 0.5, "VALE3": 0.6}}
    assert cliente.post("/carteiras/metricas", json=pedido_pesos).status_code == 422
    print("caso 422 ok; falta o caso 200")
else:
    print("rota /carteiras/metricas ainda não existe")